## 1. Import Required Libraries

In [ ]:
import os
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import yaml

# Import cosmocore modules
from cosmocore import (
    InputParams,
    compute_signal_matrix,
    read_maps,
)
from cosmocore._mpi import MPI

# Import qube modules
from picslike import PICSLike

# Set up plotting style
plt.style.use("default")
plt.rcParams["figure.figsize"] = (12, 8)

print("All imports successful!")
print(f"MPI size: {MPI.COMM_WORLD.Get_size()}, rank: {MPI.COMM_WORLD.Get_rank()}")

In [ ]:
def configure_plt():
    plt.rc("axes", labelsize=20, linewidth=1.5)
    plt.rc("xtick", direction="in", labelsize=15, top=True)
    plt.rc("ytick", direction="in", labelsize=15, right=True)

    plt.rc("xtick.major", width=1.1, size=5)
    plt.rc("ytick.major", width=1.1, size=5)

    plt.rc("xtick.minor", width=1.1, size=3)
    plt.rc("ytick.minor", width=1.1, size=3)

    plt.rc("lines", linewidth=2)
    plt.rc("legend", frameon=False, fontsize=15)
    plt.rc("figure", dpi=100, autolayout=True, figsize=[10, 7])
    plt.rc("savefig", dpi=300, bbox="tight")


configure_plt()

## 2. Setup Test Parameters and Configuration

In [ ]:
PICSLIKE_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))


def resolve_config(config_rel, overrides=None):
    """Anchor YAML paths to picslike root so the notebook runs from any cwd."""
    full_path = os.path.join(PICSLIKE_ROOT, config_rel)
    with open(full_path) as f:
        config = yaml.safe_load(f)
    for key, value in config.items():
        if isinstance(value, str):
            clean = value
            while clean.startswith("../"):
                clean = clean[3:]
            if clean.startswith("tests/") or clean.startswith("scripts/"):
                config[key] = os.path.join(PICSLIKE_ROOT, clean)
    if overrides:
        config.update(overrides)
    tmp = tempfile.NamedTemporaryFile(mode="w", suffix=".yaml", delete=False)
    yaml.dump(config, tmp, default_flow_style=False)
    tmp.close()
    return tmp.name

In [ ]:
# Set up test configuration paths
local_path = os.path.abspath(os.getcwd())
fields = "B"  # Use TQU configuration for debugging
nside = 8
# The fixture YAML defines only 3 r values, which is too sparse to see
# a likelihood peak. The theory_spectra/ directory ships 419 precomputed
# spectra across r in [0, 0.005] in 1.25e-5 increments, so we override
# the grid here to a denser linspace that hits exact filenames.
param_file = resolve_config(
    f"tests/data/nside{nside}/{fields}/fortran_reference/config.yaml",
    overrides={"parameters": {"r": [0.0, 0.005, 41]}},
)

print(f"Local path: {local_path}")
print(f"Parameter file: {param_file}")
print(f"Parameter file exists: {os.path.exists(param_file)}")

# Load parameters
params = InputParams.read_parameter_file(param_file)

# Disk-free demo: clear every output-artifact path so run() writes nothing
# (opt-in persistence, ADR-0015). Inputs are still read.
for _k in (
    "output_geometry_file",
    "outnoisecovmat1",
    "outnoisecovmat2",
    "outinvcovmatfile1",
    "outinvcovmatfile2",
    "outfilefisher",
    "outcovmatfile",
    "outerrfile",
):
    setattr(params, _k, "")

# Display key parameters
print("\n=== Key Parameters ===")
print(f"nside: {params.nside}")
print(f"spins: {params.spins}")
print(f"labels: {getattr(params, 'labels', 'Not specified')}")
print(f"physical_labels: {getattr(params, 'physical_labels', 'Not specified')}")
print(f"nspectra: {params.nspectra}")
print(f"lmax: {params.lmax}")
print(f"nsims: {params.nsims}")
print(f"do_cross: {params.do_cross}")
print(f"calibration: {params.calibration}")

## 3. Initialize Spectra Instance

In [ ]:
# Initialize Spectra instance
print("Creating Spectra instance...")
likelihood = PICSLike(params)

# Display MPI and basic info
print(f"MPI rank: {likelihood.rank}")
print(f"MPI size: {likelihood.size}")
print(f"Parameters loaded: {likelihood.params is not None}")

# Display initial variable states
print("\n=== Initial Variable States ===")
print(f"maps1: {likelihood.maps1}")

## 4. Setup Fields and Geometry

In [ ]:
# Execute setup_fields() and setup_geometry()
print("=== Setting up fields ===")
likelihood.setup_fields()
print("Fields setup completed")

print("\n=== Setting up geometry ===")
likelihood.setup_geometry()
print("Geometry setup completed")

# Inspect results
print("\n=== Geometry Results ===")
print(f"nside: {params.nside}")
print(f"Total pixels: {params.nside**2 * 12}")
print(f"npixs type: {type(likelihood.npixs)}")
print(f"npixs: {likelihood.npixs}")
print(f"n_active: {likelihood.collection.n_active}")
print(f"Total pixels: {sum(likelihood.collection.n_active)}")

if hasattr(likelihood, "pixact") and likelihood.pixact is not None:
    print(f"pixact type: {type(likelihood.pixact)}")
    print(f"pixact length: {len(likelihood.pixact)}")
    for i, pix in enumerate(likelihood.pixact):
        print(
            f"  Field {i}: {len(pix) if hasattr(pix, '__len__') else pix} active pixels"
        )

# Check collection and spectra labels
if hasattr(likelihood, "collection") and likelihood.collection is not None:
    print(f"\nSpectra labels: {likelihood.collection.spectra_labels}")
    print(f"Number of spectra: {len(likelihood.collection.spectra_labels)}")

    # This is the KEY issue we're investigating
    print('Expected spectra: ["TT", "EE", "BB", "EB", "TE", "TB"]')
    print(f"Actual spectra: {likelihood.collection.spectra_labels}")
else:
    print("Collection not available")

## 5. Load Maps Data

In [ ]:
# Execute setup_maps() - this follows the exact pattern from spectra.run()
print("=== Loading maps ===")
# Read maps using the core functionality
ntot = sum(likelihood.collection.n_active)
print(f"Total active pixels (ntot): {ntot}")

likelihood.maps1 = np.empty((ntot, likelihood.params.nsims), dtype=np.float64)
read_maps(
    maps=likelihood.maps1,
    filename=likelihood.params.inputmapfile1,
    pixact=likelihood.pixact,
    field_labels=likelihood.params.physical_labels,
    calibration=likelihood.params.calibration,
)
print("Maps loaded")

print("\n=== Maps1 Info ===")
print(f"maps1 shape: {likelihood.maps1.shape}")
print(f"maps1 dtype: {likelihood.maps1.dtype}")
print(f"maps1 min/max: {likelihood.maps1.min():.3e} / {likelihood.maps1.max():.3e}")
print(f"maps1 mean/std: {likelihood.maps1.mean():.3e} / {likelihood.maps1.std():.3e}")

## 6. Load Covariance Matrices

In [ ]:
# Load covariance matrices - from spectra.run()
print("=== Loading covariance matrices ===")
likelihood.setup_covariance_matrices()


# Load noise_cov1 (noise covariance matrix)
print(f"Loading noise_cov1 from: {likelihood.params.covmatfile1}")
likelihood.noise_cov1 = np.fromfile(likelihood.params.covmatfile1).reshape((ntot, ntot))

# Inspect covariance matrices
print("\n=== Covariance Matrix Info ===")
print(f"\nnoise_cov1 shape: {likelihood.noise_cov1.shape}")
print(f"noise_cov1 dtype: {likelihood.noise_cov1.dtype}")

## 7. Setup Cls and Beams

In [ ]:
print("=== Setting up Cls ===")
likelihood.setup_cls()
print("Cls setup completed")

print("\n=== Setting up beams ===")
likelihood.setup_beams()
print("Beams setup completed")

# Inspect the spectra collection
print("\n=== Spectra Collection Info ===")
if hasattr(likelihood, "collection") and likelihood.collection is not None:
    print(f"Collection type: {type(likelihood.collection)}")

    # Get spectra manager info
    if hasattr(likelihood.collection, "spectra_manager"):
        mgr = likelihood.collection.spectra_manager
        print(f"Spectra labels: {mgr.labels}")
        print(f"Number of spectra: {mgr.n_spectra}")

        # Show reference Cls values
        print("\n=== Reference Cls Values (first few ell) ===")
        if hasattr(mgr, "_cls_dict"):
            for label in mgr.labels:
                if label in mgr._cls_dict:
                    cls_vals = mgr._cls_dict[label]
                    print(f"{label}: {cls_vals[:5]} (ell=2-6)")

    # Get beam manager info
    if hasattr(likelihood.collection, "beam_manager"):
        print(
            f"\nBeam manager available: {likelihood.collection.beam_manager is not None}"
        )
else:
    print("Collection not available")

## 8. Smoothing factors (now internalised)

In [ ]:
# Display loaded beam (diagnostic only).
# Previously this cell built a `clnorm = b² × 2π/[ℓ(ℓ+1)]` factor that the
# parameter-scan loop multiplied into spectra by hand. PICSLike now applies
# beam smoothing internally via `collection.set_beams(lmax=lmax_signal)`,
# and `parameter_grid.get_spectrum` returns spectra already blended with
# the fiducial in the inference window, so no manual clnorm is needed.
beam_T = likelihood.collection.fields[0].beam

print(f"Beam (first 5): {beam_T[:5]}")
print(
    f"lmax_signal = {likelihood.lmax_signal}, lmax (inference) = {likelihood.params.lmax}"
)

In [ ]:
# Smoothing factors are now applied internally.
# Previously this cell built a `vecmul` array of beam-smoothing factors and stored
# it on the likelihood instance. With beam smoothing absorbed into the framework
# (collection.set_beams() in the parameter scan below applies it via apply_smoothing,
# and parameter_grid.get_spectrum returns spectra already blended with the fiducial),
# the manual vecmul is no longer needed. Kept as an explanatory marker; nothing
# downstream consumes it.
print("Smoothing factors are now applied inside collection.set_beams() — see cell below.")

## 9. Load and Process Fisher Matrix

In [ ]:
print("=== Loading Fisher matrix ===")
likelihood.setup_parameter_grid()

grid = likelihood.parameter_grid.grid_points

print(f"Parameter grid points: {grid}")

In [ ]:
from tqdm import tqdm

In [ ]:
# Parameter scan — mirrors PICSLike._compute_likelihood_point (no-basis path).
# Per parameter point: build signal matrix via collection.set_cls/set_beams +
# compute_signal_matrix, form C = N + S, invert, evaluate Gaussian likelihood.
from cosmocore.basics import matrix_inverse_symm, matrix_slogdet_symm

like_values = np.zeros(
    (
        len(likelihood.parameter_ranges["r"]),
        likelihood.params.nsims,
    )
)
chi2_values = np.zeros_like(like_values)

for i, r in enumerate(tqdm(likelihood.parameter_ranges["r"])):
    param_point = (r,)

    # parameter_grid returns the spectrum already blended with the fiducial
    # outside the inference window (parameter_grid.py:217-219).
    spectra_dict = likelihood.parameter_grid.get_spectrum(param_point)

    # set_cls + set_beams together apply beam smoothing internally
    # (fields.py:952-963 -> apply_smoothing).
    likelihood.collection.set_cls(spectra_dict, lmax=likelihood.lmax_signal)
    likelihood.collection.set_beams(lmax=likelihood.lmax_signal)

    likelihood.signal_matrix = np.zeros_like(likelihood.noise_cov1, dtype=np.float64)
    likelihood.signal_matrix = np.asfortranarray(likelihood.signal_matrix)
    compute_signal_matrix(
        S=likelihood.signal_matrix,
        lmax=likelihood.lmax_signal,
        fields=likelihood.collection,
    )

    total_cov = likelihood.noise_cov1 + likelihood.signal_matrix
    inv_total_cov = matrix_inverse_symm(total_cov)
    det = matrix_slogdet_symm(total_cov)[1]

    # chi² = d^T C^{-1} d alone is monotone in signal amplitude and does not
    # peak at the truth — parameter inference must minimize -2 log L = chi² + logdet.
    chi_squared = np.einsum(
        "in,ij,jn->n", likelihood.maps1, inv_total_cov, likelihood.maps1
    )
    neg2_log_like = chi_squared + det
    chi2_values[i] = neg2_log_like
    like_values[i] = -0.5 * neg2_log_like

In [ ]:
plt.plot(likelihood.parameter_ranges["r"], like_values[:, 0], label="Likelihood")
plt.xlabel("r")
plt.ylabel("Log-Likelihood")
plt.title("Likelihood vs r")
plt.legend()
plt.show()

plt.plot(likelihood.parameter_ranges["r"], chi2_values[:, 0], label="Chi-squared")
plt.xlabel("r")
plt.ylabel("Chi-squared")
plt.title("Chi-squared vs r")
plt.legend()
plt.show()

In [ ]:
# Equivalent to running the full likelihood in one step
print("=== Running full likelihood ===")
full_like = PICSLike(params)

full_like.run()

In [ ]:
# Run PICSLike in the harmonic computation basis (SMW formulation, no
# truncation), so this is a *basis* comparison, not a compression comparison —
# the harmonic-basis log-likelihood should agree with the full pixel-space
# path to numerical precision (per ADR-0002 vocabulary; "compression" is
# reserved for truncated bases like pixel-direct with mode_fraction < 1).
basis_config = {
    "method": "harmonic",
}

basis_like = PICSLike(params, basis=basis_config)
basis_like.run()

In [ ]:
# Compare the Gaussian log-likelihood (chi² + logdet) between the pixel-space
# path and the harmonic-basis path. They should agree to numerical precision
# since the harmonic basis is exact (no truncation). Using get_chi_squared()
# alone would drop the logdet term, which is parameter-dependent and shifts
# the peak.
pixel_negloglike = -2 * full_like.get_log_likelihood()
harmonic_negloglike = -2 * basis_like.get_log_likelihood()

plt.plot(
    full_like.parameter_ranges["r"],
    np.exp(-0.5 * (pixel_negloglike - pixel_negloglike.min())),
    label="Pixel basis (full)",
)
plt.plot(
    full_like.parameter_ranges["r"],
    np.exp(-0.5 * (harmonic_negloglike - harmonic_negloglike.min())),
    "--",
    label="Harmonic basis",
)
plt.xlabel("r")
plt.ylabel("Relative likelihood")
plt.legend()
plt.show()